# LeetCode 1598 — Crawler Log Folder bằng Python

**Link đề:** https://leetcode.com/problems/crawler-log-folder/
**Độ khó:** Easy · **Chủ đề:** Array, String, Stack

```python
class Solution:
    def minOperations(self, logs: list[str]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài toán **mô phỏng (simulation)** đường dẫn thư mục: vào thư mục con, lên thư mục cha, đứng yên.
2. Học cấu trúc dữ liệu **stack (ngăn xếp)** bằng list Python: `append`, `pop`, kiểm tra rỗng.
3. Nhận ra ta chỉ cần **độ sâu** (một số nguyên), không cần cả stack → tiết kiệm bộ nhớ `O(1)`.
4. Xử lý biên "đang ở thư mục gốc mà gặp `../`" bằng `max(0, depth - 1)`.
5. Làm quen `match-case` (Python 3.10+), trace từng bước, test đối chiếu và mở rộng sang bài Simplify Path.

## 1. Đề bài

Hệ thống file ghi lại mỗi lần người dùng **đổi thư mục**. Có 3 loại thao tác:

| Log | Ý nghĩa |
|---|---|
| `"../"` | Lên thư mục **cha**. Nếu đang ở thư mục gốc (main) thì **đứng yên**. |
| `"./"` | Ở **nguyên** thư mục hiện tại. |
| `"x/"` | Vào thư mục **con** tên `x` (luôn tồn tại). |

Bắt đầu ở thư mục gốc, thực hiện lần lượt `logs`. Trả về **số thao tác tối thiểu** để quay lại thư mục gốc.

| `logs` | Kết quả | Vì sao |
|---|:---:|---|
| `["d1/","d2/","../","d21/","./"]` | `2` | cuối cùng ở `/d1/d21` → cần 2 lần `../` |
| `["d1/","d2/","./","d3/","../","d31/"]` | `3` | cuối cùng ở `/d1/d2/d31` |
| `["d1/","../","../","../"]` | `0` | đã ở gốc (hai `../` sau không làm gì) |

**Constraints:** `1 <= logs.length <= 10^3`; `2 <= logs[i].length <= 10`; tên thư mục gồm chữ thường và chữ số.

Câu hỏi then chốt: "số thao tác tối thiểu để về gốc" chính là gì? → **độ sâu hiện tại** (mỗi `../` lên đúng 1 tầng).

In [1]:
examples = [
    (["d1/", "d2/", "../", "d21/", "./"], 2),
    (["d1/", "d2/", "./", "d3/", "../", "d31/"], 3),
    (["d1/", "../", "../", "../"], 0),
]
for logs, expected in examples:
    print(logs, "-> kỳ vọng", expected)

['d1/', 'd2/', '../', 'd21/', './'] -> kỳ vọng 2
['d1/', 'd2/', './', 'd3/', '../', 'd31/'] -> kỳ vọng 3
['d1/', '../', '../', '../'] -> kỳ vọng 0


## 2. Nền tảng Python

### 2.1. So sánh chuỗi bằng `==`

Mỗi log là một chuỗi. Ta phân loại bằng so sánh trực tiếp: `log == "../"`, `log == "./"`, còn lại là thư mục con.

Lưu ý: tên thư mục có thể chứa chữ số nhưng **không** chứa dấu chấm, nên "còn lại" chắc chắn là `x/`.

In [2]:
for log in ["../", "./", "d1/", "abc9/"]:
    if log == "../":
        kind = "lên cha"
    elif log == "./":
        kind = "đứng yên"
    else:
        kind = f"vào con '{log[:-1]}'"
    print(f"{log!r:>8} -> {kind}")

   '../' -> lên cha
    './' -> đứng yên
   'd1/' -> vào con 'd1'
 'abc9/' -> vào con 'abc9'


Ở trên có `log[:-1]`: slicing bỏ **ký tự cuối** (dấu `/`). Chỉ số âm `-1` là "phần tử cuối", `[:-1]` là "từ đầu đến trước phần tử cuối".

In [3]:
s = "d21/"
print(s[-1], "|", s[:-1], "|", len(s[:-1]))

/ | d21 | 3


### 2.2. Stack (ngăn xếp) bằng list

**Stack** hoạt động theo nguyên tắc **LIFO** — *Last In, First Out*: vào sau, ra trước (như chồng đĩa).

| Thao tác | Python | Ý nghĩa |
|---|---|---|
| push | `stack.append(x)` | đặt `x` lên đỉnh |
| pop | `stack.pop()` | lấy (và xóa) phần tử đỉnh |
| xem đỉnh | `stack[-1]` | đọc phần tử đỉnh, không xóa |
| rỗng? | `not stack` hoặc `len(stack) == 0` | |

Đường dẫn thư mục chính là một stack: vào con = push, lên cha = pop.

In [4]:
stack = []
stack.append("d1")
stack.append("d2")
print("sau 2 lần push:", stack, "| đỉnh =", stack[-1])
top = stack.pop()
print("pop ra:", top, "| còn lại:", stack)
stack.pop()
print("rỗng?", not stack, stack)

sau 2 lần push: ['d1', 'd2'] | đỉnh = d2
pop ra: d2 | còn lại: ['d1']
rỗng? True []


### 2.3. `pop()` trên list rỗng sẽ lỗi

Đây là lý do đề nói "đang ở gốc thì `../` đứng yên": ta phải **kiểm tra rỗng** trước khi `pop`.

In [5]:
empty = []
try:
    empty.pop()
except IndexError as e:
    print("IndexError:", e)
if empty:          # list rỗng -> False
    empty.pop()
print("an toàn:", empty)

IndexError: pop from empty list
an toàn: []


### 2.4. `max(0, x)` — chặn dưới bằng 0

`max(0, depth - 1)` đảm bảo độ sâu không bao giờ âm: nếu `depth = 0` thì vẫn là `0`.

In [6]:
for depth in [3, 1, 0]:
    print(f"depth={depth} -> sau '../' = {max(0, depth - 1)}")

depth=3 -> sau '../' = 2
depth=1 -> sau '../' = 0
depth=0 -> sau '../' = 0


## 3. Cách 1 — Mô phỏng bằng stack (trực giác nhất)

Giữ **đường dẫn thật** dưới dạng list tên thư mục:
- `"../"`: nếu stack không rỗng thì `pop()`.
- `"./"`: không làm gì.
- `"x/"`: `append(x)`.

Cuối cùng, đường dẫn có bao nhiêu thư mục thì cần bấy nhiêu lần `../` để về gốc → trả về `len(stack)`.

In [7]:
def min_operations_stack(logs: list[str]) -> int:
    stack = []
    for log in logs:
        if log == "../":
            if stack:
                stack.pop()
        elif log == "./":
            continue
        else:
            stack.append(log[:-1])
    return len(stack)

for logs, expected in examples:
    print(min_operations_stack(logs), "| kỳ vọng", expected)

2 | kỳ vọng 2
3 | kỳ vọng 3
0 | kỳ vọng 0


Bonus của cách stack: ta **thấy được** đường dẫn thật, rất tiện để debug.

In [8]:
def final_path(logs):
    stack = []
    for log in logs:
        if log == "../":
            if stack:
                stack.pop()
        elif log != "./":
            stack.append(log[:-1])
    return "/" + "/".join(stack)

for logs, _ in examples:
    print(final_path(logs))

/d1/d21
/d1/d2/d31
/


## 4. Nút thắt và insight — chỉ cần **độ sâu**

Đề chỉ hỏi **bao nhiêu** bước để về gốc, không hỏi tên thư mục. Ta có dùng nội dung stack ở đâu không? **Không** — chỉ dùng `len(stack)`.

Mỗi thao tác ảnh hưởng tới `len(stack)` như sau:

| Log | `len(stack)` |
|---|---|
| `"x/"` | `+1` |
| `"./"` | không đổi |
| `"../"` | `-1` nếu > 0, ngược lại giữ `0` |

→ Thay stack bằng **một biến đếm** `depth`. Bộ nhớ từ `O(n)` xuống `O(1)`.

Vì sao "tối thiểu" = `depth`? Mỗi thao tác chỉ thay đổi độ sâu tối đa 1, và `../` là cách duy nhất giảm độ sâu. Từ độ sâu `d` cần ít nhất `d` bước, và `d` lần `../` là đủ.

## 5. Lời giải chính — biến đếm `depth`

### 5.1. Đoạn 1: khởi tạo
`depth = 0` — đang ở thư mục gốc.

### 5.2. Đoạn 2: cập nhật theo từng log

In [9]:
depth = 0
for log in ["d1/", "../", "../"]:
    before = depth
    if log == "../":
        depth = max(0, depth - 1)
    elif log != "./":
        depth += 1
    print(f"{log!r:>6}: {before} -> {depth}")

 'd1/': 0 -> 1
 '../': 1 -> 0
 '../': 0 -> 0


### 5.3. Ghép lại thành hàm

In [10]:
def min_operations_depth(logs: list[str]) -> int:
    depth = 0
    for log in logs:
        if log == "../":
            depth = max(0, depth - 1)
        elif log == "./":
            pass
        else:
            depth += 1
    return depth

for logs, expected in examples:
    print(min_operations_depth(logs), "| kỳ vọng", expected)

2 | kỳ vọng 2
3 | kỳ vọng 3
0 | kỳ vọng 0


`pass` là câu lệnh "không làm gì" — dùng khi cú pháp bắt buộc có một khối lệnh. Ở đây viết ra cho rõ ba trường hợp; có thể gộp thành `elif log != "./"`.

## 6. Cách 3 — `match-case` (Python 3.10+)

`match` so khớp một giá trị với các **mẫu** (pattern). `case _:` là mẫu "mọi thứ còn lại" (giống `else`). Phù hợp khi phân loại theo vài giá trị cố định.

In [11]:
def min_operations_match(logs: list[str]) -> int:
    depth = 0
    for log in logs:
        match log:
            case "../":
                depth = max(0, depth - 1)
            case "./":
                pass
            case _:
                depth += 1
    return depth

print([min_operations_match(l) for l, _ in examples])

[2, 3, 0]


### 6.1. Một biến thể ngắn: dùng `dict` ánh xạ log → thay đổi độ sâu

`dict.get(key, default)` trả về giá trị của `key` nếu có, ngược lại trả `default`. Ta ánh xạ `"../" → -1`, `"./" → 0`, mặc định `+1`.

In [12]:
DELTA = {"../": -1, "./": 0}
print(DELTA.get("../", 1), DELTA.get("./", 1), DELTA.get("abc/", 1))

def min_operations_dict(logs: list[str]) -> int:
    depth = 0
    for log in logs:
        depth = max(0, depth + DELTA.get(log, 1))
    return depth

print([min_operations_dict(l) for l, _ in examples])

-1 0 1
[2, 3, 0]


Chú ý `max(0, depth + 0)` và `max(0, depth + 1)` không thay đổi gì vì `depth >= 0` — nên áp `max` cho mọi trường hợp là an toàn.

## 7. TRACE từng bước

Ví dụ 2: `["d1/","d2/","./","d3/","../","d31/"]`

| Bước | log | Loại | depth trước | depth sau | stack sau |
|:-:|---|---|:-:|:-:|---|
| 0 | `d1/` | vào con | 0 | 1 | `[d1]` |
| 1 | `d2/` | vào con | 1 | 2 | `[d1, d2]` |
| 2 | `./` | đứng yên | 2 | 2 | `[d1, d2]` |
| 3 | `d3/` | vào con | 2 | 3 | `[d1, d2, d3]` |
| 4 | `../` | lên cha | 3 | 2 | `[d1, d2]` |
| 5 | `d31/` | vào con | 2 | 3 | `[d1, d2, d31]` |

Kết quả `3`.

In [13]:
def trace_crawler(logs):
    depth, stack = 0, []
    print(f"{'bước':>4} | {'log':<6} | {'loại':<9} | trước | sau | stack")
    for step, log in enumerate(logs):
        before = depth
        if log == "../":
            kind = "lên cha" if stack else "ở gốc!"
            if stack:
                stack.pop()
            depth = max(0, depth - 1)
        elif log == "./":
            kind = "đứng yên"
        else:
            kind = "vào con"
            stack.append(log[:-1])
            depth += 1
        assert depth == len(stack)          # invariant
        print(f"{step:>4} | {log:<6} | {kind:<9} | {before:>5} | {depth:>3} | {stack}")
    return depth

print("Kết quả:", trace_crawler(["d1/", "d2/", "./", "d3/", "../", "d31/"]))

bước | log    | loại      | trước | sau | stack
   0 | d1/    | vào con   |     0 |   1 | ['d1']
   1 | d2/    | vào con   |     1 |   2 | ['d1', 'd2']
   2 | ./     | đứng yên  |     2 |   2 | ['d1', 'd2']
   3 | d3/    | vào con   |     2 |   3 | ['d1', 'd2', 'd3']
   4 | ../    | lên cha   |     3 |   2 | ['d1', 'd2']
   5 | d31/   | vào con   |     2 |   3 | ['d1', 'd2', 'd31']
Kết quả: 3


Trace ví dụ 3 — thấy nhánh đặc biệt "đang ở gốc mà gặp `../`":

In [14]:
print("Kết quả:", trace_crawler(["d1/", "../", "../", "../"]))

bước | log    | loại      | trước | sau | stack
   0 | d1/    | vào con   |     0 |   1 | ['d1']
   1 | ../    | lên cha   |     1 |   0 | []
   2 | ../    | ở gốc!    |     0 |   0 | []
   3 | ../    | ở gốc!    |     0 |   0 | []
Kết quả: 0


## 8. Code nộp LeetCode

### 8.1. Cách chính — biến đếm (khuyên dùng)

In [15]:
class Solution:
    def minOperations(self, logs: list[str]) -> int:
        depth = 0
        for log in logs:
            if log == "../":
                depth = max(0, depth - 1)
            elif log != "./":
                depth += 1
        return depth

sol = Solution()
for logs, expected in examples:
    assert sol.minOperations(logs) == expected
print("Solution PASS ví dụ đề")

Solution PASS ví dụ đề


### 8.2. Cách stack

In [16]:
class SolutionStack:
    def minOperations(self, logs: list[str]) -> int:
        stack = []
        for log in logs:
            if log == "../":
                if stack:
                    stack.pop()
            elif log != "./":
                stack.append(log)
        return len(stack)

print([SolutionStack().minOperations(l) for l, _ in examples])

[2, 3, 0]


### 8.3. Cách `match-case`

In [17]:
class SolutionMatch:
    def minOperations(self, logs: list[str]) -> int:
        depth = 0
        for log in logs:
            match log:
                case "../":
                    depth = max(0, depth - 1)
                case "./":
                    pass
                case _:
                    depth += 1
        return depth

print([SolutionMatch().minOperations(l) for l, _ in examples])

[2, 3, 0]


## 9. Vì sao đúng? (invariant)

**Invariant:** sau khi xử lý `k` log đầu tiên, `depth` = số tầng thư mục từ gốc tới vị trí hiện tại (= `len(stack)` trong cách 1).

- Ban đầu `depth = 0` (ở gốc) ✓.
- `x/`: đi xuống một tầng → `+1` ✓.
- `./`: không đổi ✓.
- `../`: nếu ở tầng `d > 0` thì lên `d - 1`; nếu `d = 0` thì đứng yên → `max(0, d - 1)` ✓.

**Tối thiểu:** từ tầng `d`, mỗi thao tác giảm tối đa 1 tầng nên cần ít nhất `d` thao tác; `d` lần `../` đạt đúng gốc. Vậy đáp án = `depth`.

## 10. Độ phức tạp

Gọi `n = len(logs)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Stack | `O(n)` | `O(n)` (tệ nhất toàn `x/`) |
| 2. Biến đếm `depth` | `O(n)` | `O(1)` |
| 3. `match-case` / `dict` | `O(n)` | `O(1)` |

So sánh chuỗi `log == "../"` tốn thời gian tỉ lệ độ dài chuỗi, nhưng độ dài ≤ 10 nên coi là hằng số.

## 11. Bộ test

1. Ba ví dụ đề.
2. Biên: một log mỗi loại; `../` ngay đầu tiên; toàn `./`.
3. Đi sâu rồi lên hết; lên quá gốc nhiều lần rồi đi xuống.
4. Đối chiếu 6 hàm trên log ngẫu nhiên và kiểm tra `depth == len(stack)`.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS ({actual})")

solvers = [min_operations_stack, min_operations_depth, min_operations_match, min_operations_dict,
           Solution().minOperations, SolutionStack().minOperations, SolutionMatch().minOperations]
cases = [
    ("ví dụ 1", examples[0][0], 2),
    ("ví dụ 2", examples[1][0], 3),
    ("ví dụ 3", examples[2][0], 0),
    ("một x/", ["a/"], 1),
    ("một ./", ["./"], 0),
    ("một ../ ở gốc", ["../"], 0),
    ("../ trước rồi vào", ["../", "../", "a/", "b/"], 2),
    ("toàn ./", ["./"] * 5, 0),
    ("xuống 3 lên 3", ["a/", "b/", "c/", "../", "../", "../"], 0),
    ("vượt gốc rồi xuống", ["a/", "../", "../", "../", "b9/"], 1),
    ("tên có chữ số", ["1/", "22/", "333/"], 3),
]
for name, logs, expected in cases:
    for f in solvers:
        assert f(logs) == expected, (name, f)
    check(name, min_operations_depth(logs), expected)

ví dụ 1                        PASS (2)
ví dụ 2                        PASS (3)
ví dụ 3                        PASS (0)
một x/                         PASS (1)
một ./                         PASS (0)
một ../ ở gốc                  PASS (0)
../ trước rồi vào              PASS (2)
toàn ./                        PASS (0)
xuống 3 lên 3                  PASS (0)
vượt gốc rồi xuống             PASS (1)
tên có chữ số                  PASS (3)


In [19]:
import random
rng = random.Random(1598)
for _ in range(500):
    logs = [rng.choice(["../", "./", "a/", "b1/", "zz/"]) for _ in range(rng.randint(1, 40))]
    expected = min_operations_stack(logs)
    for f in solvers:
        assert f(logs) == expected
    assert expected >= 0
print("500 dãy log ngẫu nhiên: 7 hàm khớp nhau")

worst = ["d/"] * 1000
print("1000 lần vào con ->", min_operations_depth(worst))

500 dãy log ngẫu nhiên: 7 hàm khớp nhau
1000 lần vào con -> 1000


## 12. Lỗi người mới hay gặp

### Lỗi 1: Để `depth` âm

```python
if log == "../":
    depth -= 1        # SAI ở gốc
```

Ví dụ 3: `1 → 0 → -1 → -2`, trả `-2`. **Sửa:** `max(0, depth - 1)` hoặc `if depth > 0: depth -= 1`.

In [20]:
def wrong_negative(logs):
    depth = 0
    for log in logs:
        if log == "../":
            depth -= 1
        elif log != "./":
            depth += 1
    return depth

print("sai:", wrong_negative(["d1/", "../", "../", "../"]), "| đúng:", min_operations_depth(["d1/", "../", "../", "../"]))

sai: -2 | đúng: 0


### Lỗi 2: Chỉ chặn ở **cuối** bằng `max(0, depth)`

Chặn ở cuối không đủ: khi độ sâu "âm ảo" rồi đi xuống, kết quả bị kéo thấp sai. Phải chặn **ngay mỗi bước**.

In [21]:
def wrong_clamp_end(logs):
    return max(0, wrong_negative(logs))

logs = ["../", "../", "a/", "b/"]
print("sai:", wrong_clamp_end(logs), "| đúng:", min_operations_depth(logs))

sai: 0 | đúng: 2


### Lỗi 3: Kiểm tra bằng `"." in log` hay `log.startswith(".")` rồi nhầm thứ tự

Nếu kiểm tra `log.startswith(".")` trước rồi coi là `"./"`, thì `"../"` cũng bị bắt nhầm (cũng bắt đầu bằng dấu chấm). Kiểm tra **bằng nhau đầy đủ** (`==`) hoặc kiểm tra `"../"` trước.

In [22]:
def wrong_startswith(logs):
    depth = 0
    for log in logs:
        if log.startswith("."):       # bắt cả "../" -> coi là đứng yên
            continue
        depth += 1
    return depth

print("sai:", wrong_startswith(examples[0][0]), "| đúng:", min_operations_depth(examples[0][0]))

sai: 3 | đúng: 2


### Lỗi 4: `pop()` khi stack rỗng

Cách stack mà quên `if stack:` → `IndexError` ngay ví dụ 3.

In [23]:
def wrong_pop(logs):
    stack = []
    for log in logs:
        if log == "../":
            stack.pop()
        elif log != "./":
            stack.append(log)
    return len(stack)

try:
    wrong_pop(examples[2][0])
except IndexError as e:
    print("IndexError:", e)

IndexError: pop from empty list


## 13. Mẹo nhận dạng pattern

- "Vào/ra", "mở/đóng", "đi xuống/lên" → **stack**; nếu chỉ cần **độ sâu** → **biến đếm**.
- Có giới hạn dưới (không âm, không ra khỏi gốc) → `max(0, ...)` **tại mỗi bước**.
- Đề hỏi "số bước tối thiểu để về trạng thái gốc" trong mô phỏng → thường = khoảng cách hiện tại tới gốc.
- Bài liên quan: 71 Simplify Path (stack đầy đủ với `/`), 1614 Maximum Nesting Depth of the Parentheses (biến đếm), 20 Valid Parentheses, 682 Baseball Game, 844 Backspace String Compare.

## 14. Biến thể

### 14.1. Độ sâu lớn nhất từng đạt được

In [24]:
def max_depth_reached(logs):
    depth = best = 0
    for log in logs:
        if log == "../":
            depth = max(0, depth - 1)
        elif log != "./":
            depth += 1
        best = max(best, depth)
    return best

print(max_depth_reached(examples[1][0]))   # 3

3


### 14.2. Đường dẫn tuyệt đối cuối cùng (giống Simplify Path)

Cho chuỗi đường dẫn kiểu Unix `"/a/./b/../../c/"` → `"/c"`. Tách bằng `split("/")`, bỏ phần rỗng và `"."`, `".."` thì pop.

In [25]:
def simplify_path(path: str) -> str:
    stack = []
    for part in path.split("/"):
        if part == "" or part == ".":
            continue
        if part == "..":
            if stack:
                stack.pop()
        else:
            stack.append(part)
    return "/" + "/".join(stack)

for p in ["/home/", "/../", "/a/./b/../../c/", "/home//foo/"]:
    print(f"{p!r:>20} -> {simplify_path(p)!r}")

            '/home/' -> '/home'
              '/../' -> '/'
   '/a/./b/../../c/' -> '/c'
       '/home//foo/' -> '/home/foo'


### 14.3. Số thao tác để về một thư mục **bất kỳ** cho trước

Từ đường dẫn hiện tại `A` sang đích `B`: lên tới **tổ tiên chung sâu nhất** rồi đi xuống. Số bước = `(len(A) - p) + (len(B) - p)` với `p` là độ dài tiền tố chung.

In [26]:
def steps_between(path_a: list[str], path_b: list[str]) -> int:
    p = 0
    while p < len(path_a) and p < len(path_b) and path_a[p] == path_b[p]:
        p += 1
    return (len(path_a) - p) + (len(path_b) - p)

print(steps_between(["d1", "d2", "d31"], ["d1", "x"]))   # lên 2, xuống 1 -> 3
print(steps_between(["d1", "d2"], []))                  # về gốc -> 2

3
2


## 15. Bài tập tự luyện (TODO)

**Bài 1 — tính tay.** Với `logs = ["a/", "b/", "../", "./", "c/", "d/", "../"]`, viết `depth` sau mỗi bước và đáp án.

**Bài 2 — đoán output.** Sau `s = [1, 2]; s.append(3); s.pop(); s.append(4)`, `s` là gì? `s[-1]` là gì?

### Bài 3 — Điền TODO cho lời giải biến đếm

In [27]:
def exercise_min_operations(logs: list[str]) -> int:
    # TODO 1: depth = 0
    # TODO 2: duyệt từng log; '../' -> giảm nhưng không âm; './' -> bỏ qua; còn lại -> tăng
    # TODO 3: return depth
    pass

# assert exercise_min_operations(["d1/","d2/","../","d21/","./"]) == 2

### Bài 4 — Tự viết cách stack, trả về **đường dẫn** cuối dạng chuỗi

In [28]:
def exercise_final_path(logs: list[str]) -> str:
    # TODO: trả "/" nếu ở gốc, ngược lại "/d1/d21"...
    pass

# assert exercise_final_path(["d1/","d2/","../","d21/","./"]) == "/d1/d21"
# assert exercise_final_path(["d1/","../","../"]) == "/"

### Bài 5 — Biến thể: thêm lệnh `"/"` (về thẳng gốc)

Lệnh `"/"` đưa về gốc ngay lập tức. Viết hàm trả về số `../` tối thiểu cuối cùng.

In [29]:
def exercise_with_root(logs: list[str]) -> int:
    # TODO
    pass

# assert exercise_with_root(["a/", "b/", "/", "c/"]) == 1

### Bài 6 — Mini challenge: Simplify Path tự viết

Viết lại `simplify_path` mà không xem mục 14.2.

In [30]:
def exercise_simplify_path(path: str) -> str:
    # TODO
    pass

# assert exercise_simplify_path("/a/./b/../../c/") == "/c"
# assert exercise_simplify_path("/../") == "/"

## 16. Gợi ý 3 mức

**Bài 1:** M1: `x/` +1, `./` 0, `../` −1. M2: không âm. M3: dãy bắt đầu 1, 2, 1, ...

**Bài 2:** M1: `pop` lấy phần tử **cuối**. M2: `[1,2,3]` → pop → `[1,2]`. M3: append 4.

**Bài 3:** M1: một biến đếm. M2: `max(0, depth - 1)`. M3: `elif log != "./": depth += 1`.

**Bài 4:** M1: stack lưu `log[:-1]`. M2: `if stack: stack.pop()`. M3: `"/" + "/".join(stack)` (stack rỗng → `"/"`).

**Bài 5:** M1: `"/"` đặt `depth = 0`. M2: kiểm tra `"/"` trước `else`. M3: `if log == "/": depth = 0`.

**Bài 6:** M1: `path.split("/")`. M2: bỏ `""` và `"."`. M3: `".."` → pop nếu có.

## 17. Đáp án tham khảo

**Bài 1:** `1, 2, 1, 1, 2, 3, 2` → đáp án `2`.

**Bài 2:** `s = [1, 2, 4]`, `s[-1] = 4`.

In [31]:
def answer_min_operations(logs):
    depth = 0
    for log in logs:
        if log == "../":
            depth = max(0, depth - 1)
        elif log != "./":
            depth += 1
    return depth

def answer_final_path(logs):
    stack = []
    for log in logs:
        if log == "../":
            if stack:
                stack.pop()
        elif log != "./":
            stack.append(log[:-1])
    return "/" + "/".join(stack)

def answer_with_root(logs):
    depth = 0
    for log in logs:
        if log == "/":
            depth = 0
        elif log == "../":
            depth = max(0, depth - 1)
        elif log != "./":
            depth += 1
    return depth

def answer_simplify_path(path):
    stack = []
    for part in path.split("/"):
        if part in ("", "."):
            continue
        if part == "..":
            if stack:
                stack.pop()
        else:
            stack.append(part)
    return "/" + "/".join(stack)

ex1 = ["a/", "b/", "../", "./", "c/", "d/", "../"]
assert answer_min_operations(ex1) == 2
s = [1, 2]; s.append(3); s.pop(); s.append(4)
assert s == [1, 2, 4] and s[-1] == 4
assert answer_min_operations(["d1/", "d2/", "../", "d21/", "./"]) == 2
assert answer_final_path(["d1/", "d2/", "../", "d21/", "./"]) == "/d1/d21"
assert answer_final_path(["d1/", "../", "../"]) == "/"
assert answer_with_root(["a/", "b/", "/", "c/"]) == 1
assert answer_simplify_path("/a/./b/../../c/") == "/c" and answer_simplify_path("/../") == "/"
assert answer_simplify_path("/home//foo/") == "/home/foo"
for _ in range(300):
    logs = [rng.choice(["../", "./", "a/", "b/"]) for _ in range(rng.randint(1, 30))]
    assert answer_min_operations(logs) == min_operations_stack(logs)
    fp = answer_final_path(logs)
    assert (0 if fp == "/" else fp.count("/")) == answer_min_operations(logs)
print("Đáp án tham khảo: PASS toàn bộ!")

Đáp án tham khảo: PASS toàn bộ!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minOperations(self, logs: list[str]) -> int:
        depth = 0
        for log in logs:
            if log == "../":
                depth = max(0, depth - 1)   # không lên quá gốc
            elif log != "./":
                depth += 1                  # vào thư mục con
        return depth                        # số '../' cần để về gốc
```

**Câu thần chú:** *"Vào +1, đứng 0, lên −1 nhưng không âm."*

**Checklist trước khi submit:**
- [ ] `../` ở gốc không làm `depth` âm (chặn **mỗi bước**)?
- [ ] Phân biệt đúng `"../"` và `"./"` (so sánh `==` đầy đủ)?
- [ ] Nếu dùng stack, có `if stack:` trước `pop()`?
- [ ] Đã thử ví dụ 3 (`../` liên tục ở gốc)?
- [ ] Tên method đúng `minOperations`?